In [0]:
%run "../utils/03_write_to_delta_utils"

from pyspark.sql.functions import col

#-------------------------------------------
#    Fact Low Stock Alert
#-------------------------------------------

df_fact_inventory = spark.read.table("novamart.gold.fact_inventory")
df_fact_clicks = spark.read.table("novamart.gold.fact_clicks")
df_dim_products = spark.read.table("novamart.gold.dim_products")

df_add_to_cart = (df_fact_clicks.filter(col("event_type") == "ADD_TO_CART"))

df_low_stock_alert = (df_add_to_cart
                      .join(df_fact_inventory, on = "product_id", how="inner")
                      .join(df_dim_products, on="product_id", how="inner")
                      .filter(col("stock_quantity") <= col("reorder_threshold"))
                      .select(
                          col("event_id"),
                          col("session_id"),
                          col("product_id"),
                          col("stock_quantity").alias("available_stock"),
                          col("event_at").alias("alert_timestamp")
                      ))

write_delta_table(
    df= df_low_stock_alert,
    table_name="novamart.gold.fact_low_stock_alert",
    write_mode="append",
    cluster_keys=["product_id", "alert_timestamp"]
)